In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print(tf.__version__)

2.20.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Customer_project/final_02_comp.csv"
)

df.head()

,Complaint,Product,Complaint_Length,Processed_Complaint
0,Received Capital One charge card offer XXXX. A...,Credit card,714,received capital one charge card offer applied...
1,I do n't know how they got my cell number. I t...,Debt collection,52,nt know got cell number told would deal onlybw...
2,I 'm a longtime member of Charter One Bank/RBS...,Credit card,692,longtime member charter one bankrbs citizen ba...
3,"After looking at my credit report, I saw a col...",Credit reporting,84,looking credit report saw collection account b...
4,I received a call from a XXXX XXXX from XXXX @...,Debt collection,140,received call ext stating owed wanted however ...


In [ ]:
label_encoder = joblib.load(
    "/content/drive/MyDrive/Customer_project/models/label_encoder.pkl"
)

In [ ]:
X = df["Processed_Complaint"]

y = label_encoder.transform(df["Product"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
tokenizer = Tokenizer(
    num_words=50000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
print(X_train.iloc[0])

print()

print(X_train_seq[0])

company reporting outstanding debt balance credit report charge judgment sold debt paid called barclays bank ask reporting debt outstanding charge judgment told information account longer frustrating still account reported balance charge judgment account number

[19, 58, 619, 10, 46, 3, 9, 48, 569, 237, 10, 29, 20, 1765, 7, 257, 58, 10, 619, 48, 569, 11, 17, 2, 333, 1543, 52, 2, 119, 46, 48, 569, 2, 31]


In [ ]:
vocab_size = len(tokenizer.word_index) + 1

print(vocab_size)

67027


In [ ]:
max_length = max(
    len(sequence)
    for sequence in X_train_seq
)

print(max_length)

383


In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
print(X_train_pad.shape)

print(X_test_pad.shape)

(90002, 383)
(22501, 383)


In [ ]:
# ==========================================================
# LSTM Model
# ==========================================================

model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_length,
        mask_zero=True
    )
)

# LSTM Layer
model.add(
    LSTM(
        128,
        return_sequences=False
    )
)

# Dropout
model.add(
    Dropout(0.3)
)

# Hidden Layer
model.add(
    Dense(
        64,
        activation="relu"
    )
)

# Dropout
model.add(
    Dropout(0.3)
)

# Output Layer
model.add(
    Dense(
        len(label_encoder.classes_),
        activation="softmax"
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True

)

In [ ]:
history = model.fit(

    X_train_pad,

    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=64,

    callbacks=[early_stop],

    verbose=1

)

Epoch 1/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 43s 29ms/step - accuracy: 0.6464 - loss: 1.1472 - val_accuracy: 0.7662 - val_loss: 0.8490
Epoch 2/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 32s 29ms/step - accuracy: 0.7598 - loss: 0.8090 - val_accuracy: 0.7749 - val_loss: 0.7279
Epoch 3/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.8175 - loss: 0.6292 - val_accuracy: 0.8283 - val_loss: 0.5984
Epoch 4/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 31s 28ms/step - accuracy: 0.8392 - loss: 0.5541 - val_accuracy: 0.8329 - val_loss: 0.5848
Epoch 5/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 32s 28ms/step - accuracy: 0.8716 - loss: 0.4462 - val_accuracy: 0.8390 - val_loss: 0.5588
Epoch 6/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 34s 30ms/step - accuracy: 0.8899 - loss: 0.3820 - val_accuracy: 0.8431 - val_loss: 0.5566
Epoch 7/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 38s 28ms/step - accuracy: 0.9077 - loss: 0.3227 - val_accuracy: 0.8403 - val_loss: 0.5885
Epoch 8/10
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.9183 -

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print(f"Test Accuracy: {test_accuracy:.4f}")

704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.8474 - loss: 0.5342
Test Accuracy: 0.8474


In [ ]:
y_pred_prob = model.predict(X_test_pad)

y_pred = y_pred_prob.argmax(axis=1)

704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8474


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8474
Precision: 0.8445
Recall   : 0.8474
F1 Score : 0.8449


In [ ]:
import joblib

joblib.dump(
    tokenizer,
    "/content/drive/MyDrive/Customer_project/models/lstm_tokenizer.pkl"
)

print("Tokenizer Saved Successfully!")

Tokenizer Saved Successfully!


In [ ]:
model.save(
    "/content/drive/MyDrive/Customer_project/models/lstm_model.keras"
)